# CB-SAFE &mdash; Edge-IIoTset top-up run

Edge-IIoTset data lives on Kaggle, so its experiments run here.

**What is missing.** Label-flip was run with a single seed to bound runtime, which the paper footnotes. Every configuration was later backfilled to three seeds except FedGT, which still has seed 0 only at $f = 0.1, 0.2, 0.3$. Those three cells therefore appear without error bars while every neighbouring cell carries a standard deviation.

**What this notebook does.** Runs FedGT label-flip seeds 1 and 2 at **50 rounds**, then rebuilds the tables.

**Red cells are placeholders** for numbers that exist only after the run.

## 1. Setup

Attach the Edge-IIoTset data as a Kaggle Dataset, and upload this package as a second Dataset.

In [ ]:
!pip -q install torch torchvision numpy pandas scikit-learn scipy
import os, glob, zipfile, subprocess, sys, time

CODE = '/kaggle/working/cbsafe_run'
src = glob.glob('/kaggle/input/**/cbsafe_edgeiiot_kaggle.zip', recursive=True)
assert src, 'upload cbsafe_edgeiiot_kaggle.zip as a Kaggle Dataset first'
os.makedirs(CODE, exist_ok=True)
zipfile.ZipFile(src[0]).extractall(CODE)
os.chdir(CODE)
print(sorted(os.listdir('.')))

## 2. Confirm the round count before running

`run_all_kaggle.py` defaults to **25 rounds**. The published tables use **50**. Setting `CBSAFE_ROUNDS` is what keeps this run comparable with the existing results, so check the value printed below before going on.

In [ ]:
os.environ['CBSAFE_OUT']         = '/kaggle/working/results'
os.environ['CBSAFE_DATASETS']    = 'edgeiiot'
os.environ['CBSAFE_ATTACKS']     = 'labelflip'
os.environ['CBSAFE_ROUNDS']      = '50'      # must be 50, not the default 25
os.environ['CBSAFE_OTHER_SEEDS'] = '1,2'     # seed 0 already exists

for k in ('CBSAFE_OUT','CBSAFE_DATASETS','CBSAFE_ATTACKS',
          'CBSAFE_ROUNDS','CBSAFE_OTHER_SEEDS'):
    print('%-20s %s' % (k, os.environ[k]))
assert os.environ['CBSAFE_ROUNDS'] == '50', 'round count must match the paper'

## 3. Time one configuration first

Kaggle sessions are time limited. Measure one run before committing to the grid, so the budget is known rather than guessed.

In [ ]:
t0 = time.time()
os.environ['CBSAFE_OTHER_SEEDS'] = '1'
subprocess.run([sys.executable, 'experiments/run_all_kaggle.py'], check=False)
print('elapsed for the seed-1 pass: %.1f min' % ((time.time()-t0)/60))

<div style="background:#fdecea;border-left:6px solid #c0392b;padding:10px 14px;border-radius:4px">
<b>&#9888; TO FILL IN &mdash; Runtime</b><br><br>
Minutes for the seed-1 pass: <b>________</b><br>
Projected total for seeds 1 and 2: <b>________</b><br><br>
If this will not fit in the session limit, run seed 1 now, save the output, and do seed 2 in a second session. The runner skips configurations whose CSV already exists, so re-running is safe.
</div>

## 4. Run the remaining seed

In [ ]:
os.environ['CBSAFE_OTHER_SEEDS'] = '2'
subprocess.run([sys.executable, 'experiments/run_all_kaggle.py'], check=False)

got = sorted(glob.glob('/kaggle/working/results/kaggle/edgeiiot/robust_labelflip_fedgt_*'))
print('FedGT label-flip files now present:')
for g in got: print('  ', os.path.basename(g))

<div style="background:#fdecea;border-left:6px solid #c0392b;padding:10px 14px;border-radius:4px">
<b>&#9888; TO FILL IN &mdash; Coverage after the run</b><br><br>
FedGT label-flip files present: <b>____ of 9</b> (3 values of $f$ &times; 3 seeds)<br>
Any configuration that failed: <b>________________</b>
</div>

## 5. Read off the numbers

Final-round accuracy per configuration, which is what the table cell reports.

In [ ]:
import pandas as pd, numpy as np, collections
rows = collections.defaultdict(list)
for p in glob.glob('/kaggle/working/results/kaggle/edgeiiot/robust_labelflip_fedgt_*.csv'):
    df = pd.read_csv(p)
    f = os.path.basename(p).split('_f')[1][:2]
    rows[f].append(100*df['acc'].iloc[-1])
for f in sorted(rows):
    v = np.array(rows[f])
    print('f=0.%s  n=%d  acc = %.1f +/- %.1f' %
          (f[0] if f[1]=='0' else f, len(v), v.mean(),
           v.std(ddof=1) if len(v) > 1 else 0.0))

<div style="background:#fdecea;border-left:6px solid #c0392b;padding:10px 14px;border-radius:4px">
<b>&#9888; TO FILL IN &mdash; FedGT label-flip on Edge-IIoTset</b><br><br>
f = 0.1: <b>______ &plusmn; ______</b> &nbsp; (paper currently: 62.0, no error bar)<br>
f = 0.2: <b>______ &plusmn; ______</b> &nbsp; (paper currently: 62.7, no error bar)<br>
f = 0.3: <b>______ &plusmn; ______</b> &nbsp; (paper currently: 62.0, no error bar)<br><br>
These replace the three single-seed cells in the Edge-IIoTset row of the label-flip table, and let the single-seed footnote be dropped.
</div>

## 6. Rebuild the tables

In [ ]:
subprocess.run([sys.executable, 'experiments/build_tables.py'], check=False)
for p in glob.glob('/kaggle/working/results/tables/*.tex'):
    print('---', os.path.basename(p))
    print(open(p).read()[:600])

<div style="background:#fdecea;border-left:6px solid #c0392b;padding:10px 14px;border-radius:4px">
<b>&#9888; TO FILL IN &mdash; Did the numbers move?</b><br><br>
Did the mean shift once three seeds were averaged? <b>yes / no</b><br>
If a mean moved by more than about one point, the single-seed value was not representative, which is worth a sentence in the paper.
</div>

## 7. Save the output

Kaggle discards `/kaggle/working` when the session ends unless output is saved. Run this before the session times out, then download `results/kaggle/edgeiiot/` and drop it into the repository at the same path.

In [ ]:
!cd /kaggle/working && zip -qr edgeiiot_topup.zip results/kaggle/edgeiiot results/tables
!ls -la /kaggle/working/edgeiiot_topup.zip